# SQL Generator Assistant — End-to-End Pipeline Execution

Run the **complete agentic loop** from dataset loading through MEP generation,
evaluation, failure taxonomy, and summary — all from one notebook.

**How to use:** Edit the parameters in **Cell 1 (Configuration)**, then *Run All*.

| Section | What it does |
|---|---|
| 1 — Configuration | All tunable parameters in one place |
| 2 — Environment | Check API keys, install path, imports |
| 2.5 — Langfuse health check | Verify Langfuse credentials are configured before running |
| 3 — Database Setup | Load UCI credit card CSV into SQLite |
| 4 — Load dataset | Pull samples from eval_samples.json |
| 5 — Instantiate agents | Build Planner, SchemaRetriever, SQLGenerator, Verifier |
| 6 — Run pipeline | Generate MEPs (Plan → Schema → SQL → Verify) |
| 7 — Inspect MEP | Browse the first result |
| 8 — Schema Retriever insight | See what KPI/tables were matched |
| 9 — Ablation: no schema retriever | Re-run without SchemaRetriever to compare |
| 10 — Pass 1: eval outputs | Rule-based accuracy + optional LLM judge |
| 11 — Pass 2: eval traces | Latency, SQL calls, replayability |
| 12 — Pass 3: top-K evaluation | hit@1 / hit@2 / hit@3 |
| 13 — Pass 4: failure taxonomy | SQL-aware diagnosis of wrong answers |
| 14 — Summarize | Aggregate to summary.csv |

> **Companion notebook:** `analysis.ipynb` visualises existing results with charts and dashboards.
> This notebook *generates* the results.

## 1 — Configuration

Change these values. Everything else runs automatically.

In [1]:
import sys
import uuid
from pathlib import Path


# ── Tune these before running ────────────────────────────────────────────────
N_SAMPLES    = 1      # samples to process (keep ≤10 for a first run)
CONFIG       = "anthropic_claude"  # anthropic_claude | openai_openai | gemini_gemini | openai_gemini | gemini_openai
USE_VERIFIER = True   # False = skip VerifierAgent (--no_verifier equivalent)
USE_JUDGE    = False  # True = LLM rubric judge (extra API calls per sample)
RUN_ABLATION = True   # True = also run without SchemaRetriever after main run
# ────────────────────────────────────────────────────────────────────

REPO_ROOT    = Path(".").resolve()
RUN_ID       = str(uuid.uuid4())
OUT_DIR      = str(REPO_ROOT / "meps" / CONFIG / "sql_eval")
SAMPLES_PATH = str(REPO_ROOT / "src" / "agentic_chartqapro_eval" / "eval" / "eval_samples.json")
CSV_PATH     = str(REPO_ROOT / "data" / "default of credit card clients.xls")
DB_PATH      = str(REPO_ROOT / "data" / "credit_card_clients.db")
OUT_LABEL    = f"{CONFIG}_n{N_SAMPLES}"

ver_status = "enabled" if USE_VERIFIER else "disabled"
print(f"Config     : {CONFIG}")
print(f"Samples    : {N_SAMPLES}")
print(f"Verifier   : {ver_status}")
print(f"LLM judge  : {'enabled' if USE_JUDGE else 'disabled (--no_judge)'}")
print(f"Run ID     : {RUN_ID}")
print(f"Output dir : {OUT_DIR}")

## 2 — Environment

Verify API keys and import all modules. Fix any errors here before proceeding.

In [4]:
import json
import os


# Make sure src/ is on the path
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from dotenv import load_dotenv  # noqa: E402


load_dotenv(REPO_ROOT / ".env")

# API key check — each backend needs its own key
KEY_MAP = {
    "ANTHROPIC_API_KEY": "anthropic",
    "OPENAI_API_KEY":    "openai",
    "GEMINI_API_KEY":    "gemini",
}
missing = []
for var, needed_for in KEY_MAP.items():
    val = os.environ.get(var, "")
    needed = needed_for in CONFIG
    if val and not val.startswith("your_"):
        print(f"  ok  {var}  ({val[:3]}...)")
    elif needed:
        print(f"  MISSING  {var}  <- required for {CONFIG}")
        missing.append(var)
    else:
        print(f"  skip  {var}  (not needed for {CONFIG})")

if missing:
    raise EnvironmentError(f"Set {missing} in .env and re-run this cell.")

# Core imports
import pandas as pd  # noqa: E402
from agentic_chartqapro_eval.agents.planner_agent import PlannerAgent  # noqa: E402
from agentic_chartqapro_eval.agents.sql_retrieval_agent import SQLRetrievalAgent  # noqa: E402
from agentic_chartqapro_eval.agents.sqlgenerator_agent import SQLGeneratorAgent  # noqa: E402
from agentic_chartqapro_eval.agents.verifier_agent import VerifierAgent  # noqa: E402
from agentic_chartqapro_eval.eval.db_setup import setup_db  # noqa: E402
from agentic_chartqapro_eval.eval.error_taxonomy import classify_failure  # noqa: E402
from agentic_chartqapro_eval.eval.eval_outputs import evaluate_mep  # noqa: E402
from agentic_chartqapro_eval.eval.eval_topk import evaluate_topk  # noqa: E402
from agentic_chartqapro_eval.eval.eval_traces import evaluate_trace  # noqa: E402
from agentic_chartqapro_eval.eval.summarize import summarize, write_csv  # noqa: E402
from agentic_chartqapro_eval.langfuse_integration.client import get_client  # noqa: E402
from agentic_chartqapro_eval.mep.writer import iter_meps  # noqa: E402
from agentic_chartqapro_eval.runner.run_generate_meps import (  # noqa: E402
    BACKEND_CONFIGS,
    process_sample,
)

print("\nAll imports OK")

## 2.5 — Langfuse Health Check

Verifies that Langfuse credentials are configured before the pipeline runs.

| Check | What it tests |
|---|---|
| Env vars present | `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY` are set in `.env` |
| Client init | `Langfuse()` initialises without error |

If the keys are absent the cell prints a skip notice and continues — Langfuse is optional.
The pipeline produces identical MEPs with or without it; tracing is purely additive.

In [5]:
from agentic_chartqapro_eval.langfuse_integration.client import reset_client


# Force re-initialisation so re-running this cell picks up any .env changes
reset_client()

lf_public = os.environ.get("LANGFUSE_PUBLIC_KEY", "")
lf_secret = os.environ.get("LANGFUSE_SECRET_KEY", "")

if not lf_public or not lf_secret:
    print("[skip] LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY are not set.")
    print("       Langfuse tracing is disabled. Pipeline will run fine without it.")
    print()
    print("To enable Langfuse tracing, add to .env:")
    print("  LANGFUSE_PUBLIC_KEY=pk-lf-...")
    print("  LANGFUSE_SECRET_KEY=sk-lf-...")
    print("  # LANGFUSE_HOST=https://cloud.langfuse.com  (default; change for self-hosted)")
else:
    results = {}

    # -- Check 1: Env vars present --
    results["env"] = ("ok", f"pk={lf_public[:3]}...")

    # -- Check 2: Client initialises --
    try:
        _lf_hc = get_client()
        if _lf_hc is not None:
            results["client"] = ("ok", "Langfuse() ready")
        else:
            results["client"] = ("fail", "get_client() returned None")
    except Exception as e:
        results["client"] = ("fail", str(e))

    # -- Report --
    labels = [
        ("env", "Env vars present"),
        ("client", "Client init     "),
    ]
    all_ok = True
    for key, label in labels:
        status, detail = results.get(key, ("skip", ""))
        marker = "✓ OK  " if status == "ok" else ("⊘ skip" if status == "skip" else "✗ FAIL")
        if status not in ("ok", "skip"):
            all_ok = False
        print(f"  {marker}  {label}  {detail}")

    print()
    if all_ok:
        lf_host = os.environ.get("LANGFUSE_HOST") or os.environ.get("LANGFUSE_BASE_URL") or "https://cloud.langfuse.com"
        print("✓ Langfuse is configured.")
        print(f"Host      : {lf_host}")
        print("Traces and scores will be recorded automatically during the pipeline run.")
    else:
        print("⚠ WARNING: Langfuse client failed to initialise.")
        print("The pipeline will still run; tracing will be skipped.")

## 3 — Database Setup

Load the UCI credit card CSV into a local SQLite database.
This runs in ~1 s and is safe to re-run (`if_exists="replace"`).

Both `SQLGeneratorAgent` and `VerifierAgent` need the `db_uri` returned here.

In [ ]:
db_uri = setup_db(CSV_PATH, DB_PATH)
print(f"\nDB URI: {db_uri}")

## 4 — Load Dataset

Loads samples from `eval_samples.json` (flat JSON — no HuggingFace download required).

In [ ]:
from collections import Counter


with open(SAMPLES_PATH) as _f:
    samples = json.load(_f)

if N_SAMPLES:
    samples = samples[:N_SAMPLES]

print(f"Loaded {len(samples)} samples from {SAMPLES_PATH}\n")

qt_counts = Counter(s["question_type"] for s in samples)
print("Question type breakdown:")
for qt, n in sorted(qt_counts.items()):
    print(f"  {qt:<22} {n}")

s0 = samples[0]
print(f"\nFirst sample: {s0['sample_id']}")
print(f"  Q   : {s0['question']}")
print(f"  A   : {s0['expected_output']}")
print(f"  KPI : {s0.get('kpi_name', '—')}")

## 5 — Instantiate Agents

Build the four components of the pipeline. Agents are created once and reused across all samples.

In [ ]:
config = dict(BACKEND_CONFIGS[CONFIG])
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# Stage 1: PlannerAgent — text-only LLM, identifies KPI intent
planner = PlannerAgent(
    backend=config["planner_backend"],
    model=config["planner_model"],
)
print(f"PlannerAgent       : {config['planner_backend']} / {config['planner_model']}")

# Stage 2: SQLRetrievalAgent — maps question to KPI registry
schema_retriever = SQLRetrievalAgent(
    backend=config["sql_backend"],
    model=config["sql_model"],
)
print(f"SQLRetrievalAgent  : {config['sql_backend']} / {config['sql_model']}")

# Stage 3: SQLGeneratorAgent — NL → SQL via NL2SQLTool
sql_generator = SQLGeneratorAgent(
    db_uri=db_uri,
    backend=config["sql_backend"],
    model=config["sql_model"],
)
print(f"SQLGeneratorAgent  : {config['sql_backend']} / {config['sql_model']}  (db={DB_PATH})")

# Stage 4: VerifierAgent (optional) — re-checks SQL logic against KPI definitions
verifier = None
if USE_VERIFIER:
    verifier = VerifierAgent(
        db_uri=db_uri,
        backend=config["sql_backend"],
        model=config["sql_model"],
    )
    print(f"VerifierAgent      : {config['sql_backend']} / {config['sql_model']}")
else:
    print("VerifierAgent      : disabled (USE_VERIFIER=False)")

# Langfuse observability (no-op if keys not set)
lf_client = get_client()
lf_status = "enabled" if lf_client else "not configured"
print(f"Langfuse           : {lf_status}")

# MEPConfig for process_sample
from agentic_chartqapro_eval.eval.eval_runner import make_config  # noqa: E402
mep_config = make_config(
    backend=config["sql_backend"],
    model=config["sql_model"],
    config_name=CONFIG,
    schema_retriever_enabled=True,
    verifier_enabled=USE_VERIFIER,
)

## 6 — Run the Pipeline

For each sample: **Plan → Schema → SQL → [Verify] → write MEP**.

Each call to `process_sample()` is fully self-contained and thread-safe.
To parallelise, increase `N_WORKERS` below.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


N_WORKERS = 1  # set >1 to parallelise (increase N_SAMPLES too)

mep_paths = []
print(f"Running {len(samples)} samples  workers={N_WORKERS}  -> {OUT_DIR}\n")


def _run_one(sample):
    return process_sample(
        sample,
        planner=planner,
        schema_retriever=schema_retriever,
        sql_generator=sql_generator,
        verifier_agent=verifier,
        config=config,
        mep_config=mep_config,
        run_id=RUN_ID,
        out_dir=OUT_DIR,
        lf_client=lf_client,
    )


if N_WORKERS <= 1:
    for i, sample in enumerate(samples, 1):
        print(f"[{i}/{len(samples)}] {sample['sample_id']} ...", end=" ", flush=True)
        try:
            path = _run_one(sample)
            mep_paths.append(path)
            print("OK")
        except Exception as exc:
            print(f"ERROR: {exc}")
else:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(_run_one, s): s for s in samples}
        for done, fut in enumerate(as_completed(futures), 1):
            s = futures[fut]
            try:
                path = fut.result()
                mep_paths.append(path)
                print(f"[{done}/{len(samples)}] {s['sample_id']} -> OK")
            except Exception as exc:
                print(f"[{done}/{len(samples)}] {s['sample_id']} ERROR: {exc}")

print(f"\nDone -- {len(mep_paths)}/{len(samples)} MEPs written to {OUT_DIR}")

## 7 — Inspect First MEP

MEPs are self-contained JSON files. Every field you see here is what the agent
actually produced — no post-processing.

The `lf_trace_id` links this MEP back to the live trace in the Langfuse dashboard
if Langfuse is configured.

In [ ]:
if not mep_paths:
    print("No MEPs written -- check errors above.")
else:
    mep = json.loads(Path(sorted(mep_paths)[0]).read_text())
    s          = mep["sample"]
    pl         = mep.get("plan", {}).get("parsed", {})
    sql_gen    = mep.get("sql_generator", {}) or {}
    sql_parsed = sql_gen.get("parsed", {}) or {}
    schema_ret = mep.get("schema_retriever") or {}
    vr         = (mep.get("verifier") or {}).get("parsed", {})
    ts         = mep.get("timestamps", {})

    print("=" * 64)
    print(f"Sample   : {s['sample_id']}")
    print(f"Type     : {s['question_type']}")
    print(f"Question : {s['question']}")
    print(f"Expected : {s['expected_output']}")
    print()
    print("Planner steps:")
    for j, step in enumerate(pl.get("steps", []), 1):
        print(f"  {j}. {step}")
    print()
    if schema_ret:
        print(f"Schema retriever KPI   : {schema_ret.get('kpi_name', '—')}")
        print(f"  Source tables        : {schema_ret.get('source_tables', [])}")
        print(f"  Source fields        : {schema_ret.get('source_fields', [])}")
    print()
    sql_q = sql_parsed.get("sql", "—")
    print(f"SQL query    : {sql_q[:120]}")
    print(f"Answer       : {sql_parsed.get('answer', '—')}")
    print(f"Explanation  : {str(sql_parsed.get('explanation', '—'))[:200]}")
    print(f"Source tables: {sql_parsed.get('source_tables', [])}")
    print(f"Citation OK  : {bool(sql_parsed.get('source_tables'))}")
    print(f"Guardrail    : {sql_parsed.get('guardrail_triggered', False)}")
    if vr:
        print()
        print(f"Verifier verdict  : {vr.get('verdict', '—')}")
        print(f"Verifier answer   : {vr.get('answer', '—')}")
        print(f"Verifier reasoning: {str(vr.get('reasoning', '—'))[:200]}")
    print()
    print("Timestamps (ms):")
    for k in ["planner_ms", "schema_retriever_ms", "sql_generator_ms", "verifier_ms"]:
        print(f"  {k:<28} {ts.get(k, 0):.0f}")
    if mep.get("lf_trace_id"):
        print(f"Langfuse trace ID: {mep['lf_trace_id']}")
    print("=" * 64)

## 8 — Schema Retriever Insight: What Was Matched to the KPI Registry?

The `SQLRetrievalAgent` makes a single LLM call to map the user's question
onto an approved KPI definition from the registry.

Its structured output — `kpi_name`, `source_tables`, `source_fields`, `join_keys`,
`metric_logic` — is injected into the `SQLGeneratorAgent`'s prompt as context.

When `schema_retriever_enabled=False`, this context is absent and the SQL
generator must infer table/column names from the question alone.

In [ ]:
if not mep_paths:
    print("No MEPs to inspect.")
else:
    mep        = json.loads(Path(sorted(mep_paths)[0]).read_text())
    schema_ret = mep.get("schema_retriever") or {}

    if not schema_ret:
        print("Schema retriever was disabled or returned no result.")
        print("Re-run with schema_retriever_enabled=True to see output.")
    else:
        ts = mep.get("timestamps", {})
        print(f"Schema retriever elapsed : {ts.get('schema_retriever_ms', 0):.0f} ms")
        print(f"Parse error              : {schema_ret.get('parse_error', False)}")
        print()
        print("Structured schema retriever output:")
        print(f"  kpi_name      : {schema_ret.get('kpi_name', '—')}")
        print(f"  source_tables : {schema_ret.get('source_tables', [])}")
        print(f"  source_fields : {schema_ret.get('source_fields', [])}")
        print(f"  join_keys     : {schema_ret.get('join_keys', [])}")
        print(f"  metric_logic  : {schema_ret.get('metric_logic', '—')}")
        print()
        raw = schema_ret.get("raw_text", "")
        print("Raw schema retriever response (first 400 chars):")
        print(raw[:400])

## 9 — Ablation: Run Without Schema Retriever

Runs the same samples with `schema_retriever_enabled=False` and writes to a
separate directory. This lets you compare the two runs side-by-side in Pass 1
evaluation.

Set `RUN_ABLATION=False` in the config cell to skip this section.

In [ ]:
no_schema_paths = []
OUT_DIR_NO_SCHEMA = str(REPO_ROOT / "meps" / (CONFIG + "_no_schema") / "sql_eval")
RUN_ID_NO_SCHEMA  = str(uuid.uuid4())

if not RUN_ABLATION:
    print("Skipped (RUN_ABLATION=False).")
else:
    mep_config_no_schema = make_config(
        backend=config["sql_backend"],
        model=config["sql_model"],
        config_name=CONFIG + "_no_schema",
        schema_retriever_enabled=False,
        verifier_enabled=USE_VERIFIER,
    )
    Path(OUT_DIR_NO_SCHEMA).mkdir(parents=True, exist_ok=True)
    print(f"Running {len(samples)} samples WITHOUT SchemaRetriever -> {OUT_DIR_NO_SCHEMA}\n")
    for i, sample in enumerate(samples, 1):
        print(f"[{i}/{len(samples)}] {sample['sample_id']} ...", end=" ", flush=True)
        try:
            path = process_sample(
                sample,
                planner=planner,
                schema_retriever=schema_retriever,   # agent passed but disabled via config
                sql_generator=sql_generator,
                verifier_agent=verifier,
                config=config,
                mep_config=mep_config_no_schema,
                run_id=RUN_ID_NO_SCHEMA,
                out_dir=OUT_DIR_NO_SCHEMA,
                lf_client=lf_client,
            )
            no_schema_paths.append(path)
            print("OK")
        except Exception as exc:
            print(f"ERROR: {exc}")
    print(f"\nDone -- {len(no_schema_paths)} no-schema MEPs written")
    if mep_paths and no_schema_paths:
        mep_with    = json.loads(Path(sorted(mep_paths)[0]).read_text())
        mep_without = json.loads(Path(no_schema_paths[0]).read_text())
        ts_with     = mep_with.get("timestamps", {})
        ts_without  = mep_without.get("timestamps", {})
        print("\nSchema retriever vs no-schema:")
        print(f"  With schema    schema_retriever_ms={ts_with.get('schema_retriever_ms', 0):.0f}  sql_generator_ms={ts_with.get('sql_generator_ms', 0):.0f}")
        print(f"  Without schema schema_retriever_ms={ts_without.get('schema_retriever_ms', 0):.0f}  sql_generator_ms={ts_without.get('sql_generator_ms', 0):.0f}")

## 10 — Pass 1: Evaluate Outputs

Rule-based accuracy scoring + optional LLM judge (5 rubric dimensions).

- `answer_accuracy`: exact-match with numeric tolerance
- `citation_present`: whether `source_tables` is non-empty
- `guardrail_triggered`: whether a blocked query was detected
- `verifier_verdict`: `confirmed` / `revised` / `skipped`
- `judge_*`: scored 0–1 by a text LLM (only when `USE_JUDGE=True`)

Results are written to `metrics_<label>.jsonl` in the output directory.

In [ ]:
metrics_path = str(REPO_ROOT / "output" / f"metrics_{OUT_LABEL}.jsonl")
Path(REPO_ROOT / "output").mkdir(parents=True, exist_ok=True)
metrics_list = []

print(f"Evaluating MEPs in {OUT_DIR} ...")
with open(metrics_path, "w") as f_out:
    for mep_dict in iter_meps(OUT_DIR):
        try:
            m = evaluate_mep(
                mep_dict,
                use_judge=USE_JUDGE,
                judge_backend=config.get("judge_backend", "anthropic"),
                judge_model=config.get("sql_model", "claude-sonnet-4-6"),
            )
            f_out.write(json.dumps(m) + "\n")
            metrics_list.append(m)
        except Exception as exc:
            print(f"  eval error: {exc}")

print(f"Pass 1 complete -- {len(metrics_list)} rows -> {metrics_path}\n")

df = pd.DataFrame(metrics_list)
print(f"Overall accuracy  : {df['answer_accuracy'].mean():.1%}  (n={len(df)})")
if "citation_present" in df.columns:
    print(f"Citation present  : {df['citation_present'].mean():.1%}")
if "guardrail_triggered" in df.columns:
    print(f"Guardrail hits    : {df['guardrail_triggered'].sum()}/{len(df)}")
if "verifier_verdict" in df.columns and not df["verifier_verdict"].eq("skipped").all():
    revised = (df["verifier_verdict"] == "revised").sum()
    print(f"Verifier revised  : {revised}/{len(df)} ({revised / len(df):.1%})")

print()
print("Accuracy by question type:")
for qt, grp in df.groupby("question_type"):
    print(f"  {qt:<22} {grp['answer_accuracy'].mean():.1%}  (n={len(grp)})")

In [ ]:
# Compare schema retriever vs no-schema accuracy (only runs if ablation was done)
if no_schema_paths:
    no_schema_metrics = []
    for mep_dict in iter_meps(OUT_DIR_NO_SCHEMA):
        import contextlib
        with contextlib.suppress(Exception):
            no_schema_metrics.append(evaluate_mep(mep_dict, use_judge=False))
    df_no_schema = pd.DataFrame(no_schema_metrics)
    print("Accuracy comparison:")
    print(f"  WITH schema retriever    : {df['answer_accuracy'].mean():.1%}  (n={len(df)})")
    print(f"  WITHOUT schema retriever : {df_no_schema['answer_accuracy'].mean():.1%}  (n={len(df_no_schema)})")
    delta = df["answer_accuracy"].mean() - df_no_schema["answer_accuracy"].mean()
    print(f"  Delta                    : {delta:+.1%}  (positive = schema retriever helped)")
else:
    print("Ablation not run -- set RUN_ABLATION=True and re-run to compare.")

## 11 — Pass 2: Evaluate Traces

Trace-level metrics: latency per agent, SQL tool call counts, and replayability score
(fraction of required MEP fields that are present and non-empty).

In [ ]:
trace_path = str(REPO_ROOT / "output" / f"trace_metrics_{OUT_LABEL}.jsonl")
Path(REPO_ROOT / "output").mkdir(parents=True, exist_ok=True)
trace_rows = []

with open(trace_path, "w") as f_out:
    for mep_dict in iter_meps(OUT_DIR):
        row = evaluate_trace(mep_dict)
        f_out.write(json.dumps(row) + "\n")
        trace_rows.append(row)

trace_df = pd.DataFrame(trace_rows)
print(f"Pass 2 complete -- {len(trace_df)} rows -> {trace_path}\n")

cols = [
    "latency_sec",
    "planner_latency_sec",
    "schema_retriever_latency_sec",
    "sql_generator_latency_sec",
    "sql_call_count",
    "replayability",
    "error_count",
]
cols = [c for c in cols if c in trace_df.columns]
print(trace_df[cols].describe().round(3).to_string())

## 12 — Pass 3: Top-K Evaluation (hit@1 / hit@2 / hit@3)

Re-queries the SQLGeneratorAgent K times with a temperature ladder (0.2 → 0.4 → 0.6),
executes each SQL against the real DB, deduplicates candidates, and computes hit@k:
whether the expected answer appears within the top-k candidates.

- **hit@1** ≈ standard accuracy with a fresh independent call
- **hit@2/3** measures how often the right answer is in the model's top candidates

In [ ]:
topk_path = str(REPO_ROOT / "output" / f"topk_{OUT_LABEL}.jsonl")
Path(REPO_ROOT / "output").mkdir(parents=True, exist_ok=True)
topk_list = []

K            = 3
topk_backend = config.get("sql_backend", "anthropic")
topk_model   = config.get("sql_model",   "claude-sonnet-4-6")

print(f"Running top-{K} evaluation on MEPs in {OUT_DIR} ...")
print(f"  backend={topk_backend}  model={topk_model}\n")

with open(topk_path, "w") as f_out:
    for mep_dict in iter_meps(OUT_DIR):
        try:
            result = evaluate_topk(
                mep_dict,
                k=K,
                backend=topk_backend,
                model=topk_model,
                db_uri=db_uri,
            )
            f_out.write(json.dumps(result) + "\n")
            topk_list.append(result)
            sid   = result["sample_id"]
            cands = result["topk_candidates"]
            h1    = result.get("hit_at_1", 0)
            hk    = result.get(f"hit_at_{K}", 0)
            print(f"  {sid}  candidates={cands}  hit@1={h1:.0f}  hit@{K}={hk:.0f}")
        except Exception as exc:
            print(f"  eval error: {exc}")

print(f"\nTop-{K} complete -- {len(topk_list)} rows -> {topk_path}\n")

if topk_list:
    orig_acc = sum(r["original_accuracy"] for r in topk_list) / len(topk_list)
    print(f"Original accuracy (from MEP)  : {orig_acc:.1%}")
    for ki in range(1, K + 1):
        key = f"hit_at_{ki}"
        val = sum(r.get(key, 0) for r in topk_list) / len(topk_list)
        print(f"hit@{ki}                         : {val:.1%}")

## 13 — Pass 4: Failure Taxonomy

For each **wrong** answer, an LLM examines the SQL query, the expected output,
and the agent's explanation — and classifies the primary failure mode.

SQL failure categories: `wrong_table`, `wrong_aggregation`, `wrong_filter`,
`date_range_error`, `join_error`, `metric_definition_mismatch`, `guardrail_blocked`,
`parse_failure`, `unanswerable_failure`, `question_misunderstanding`, `other`.

Guardrail-blocked and parse-failure samples short-circuit without an LLM call.
Correct answers (`answer_accuracy == 1.0`) are skipped.

In [ ]:
taxonomy_path = str(REPO_ROOT / "output" / f"taxonomy_{OUT_LABEL}.jsonl")
Path(REPO_ROOT / "output").mkdir(parents=True, exist_ok=True)
taxonomy_rows = []

# Build accuracy lookup from Pass 1
acc_map = {r["sample_id"]: r["answer_accuracy"] for r in metrics_list}

tax_backend = config.get("sql_backend", "anthropic")
tax_model   = config.get("sql_model",   "claude-sonnet-4-6")

print(f"Classifying failures using {tax_backend} / {tax_model} ...")
with open(taxonomy_path, "w") as f_out:
    for mep_dict in iter_meps(OUT_DIR):
        sid = mep_dict.get("sample", {}).get("sample_id", "")
        acc = acc_map.get(sid, 0.0)
        try:
            result = classify_failure(
                mep_dict,
                answer_accuracy=acc,
                backend=tax_backend,
                model=tax_model,
            )
            row = {"sample_id": sid, "answer_accuracy": acc, **result}
            f_out.write(json.dumps(row) + "\n")
            taxonomy_rows.append(row)
            marker = "(skipped -- correct)" if acc == 1.0 else result.get("failure_type", "")
            print(f"  {sid}: {marker}")
        except Exception as exc:
            print(f"  {sid}: ERROR {exc}")

tax_df = pd.DataFrame(taxonomy_rows)
print(f"\nPass 4 complete -- {len(tax_df)} rows -> {taxonomy_path}\n")
print("Failure type breakdown:")
print(tax_df["failure_type"].value_counts().to_string())

## 14 — Summarize

Aggregate all metrics into `summary_<label>.csv`, grouped by `config_name` and `question_type`.
To compare multiple configs, run cells 6–13 again with a different `CONFIG` value
then concatenate the `metrics_*.jsonl` files before summarising.

In [ ]:
summary_path = str(REPO_ROOT / "output" / f"summary_{OUT_LABEL}.csv")
Path(REPO_ROOT / "output").mkdir(parents=True, exist_ok=True)

rows = summarize(metrics_list)
write_csv(rows, summary_path)
print(f"Summary written -> {summary_path}  ({len(rows)} rows)\n")

from IPython.display import display  # noqa: A004, E402

summary_df = pd.DataFrame(rows)
rename = {
    "answer_accuracy_mean": "accuracy",
    "latency_sec_mean":     "latency_s",
    "sql_call_count_mean":  "sql_calls",
}
display(
    summary_df.rename(columns=rename)[[
        c for c in [
            "config_name", "question_type", "count",
            "accuracy", "latency_s", "sql_calls",
        ] if c in summary_df.rename(columns=rename).columns
    ]]
    .fillna("")
    .style.format({"accuracy": "{:.1%}", "latency_s": "{:.1f}", "sql_calls": "{:.1f}"})
)

## What to Explore Next

- **`analysis.ipynb`** — visualise the results you just generated: accuracy charts, verifier
  revision analysis, failure taxonomy bar chart, judge score distributions, per-sample browser

- **Compare configs** — change `CONFIG` in Cell 1 to `openai_openai` or `gemini_gemini`,
  re-run cells 6–13, then concatenate the two `metrics_*.jsonl` files and summarise together

- **Streamlit dashboard** — `streamlit run src/agentic_chartqapro_eval/eval/dashboard.py`

- **HTML report** — `uv run --env-file .env -m agentic_chartqapro_eval.eval.report --metrics output/metrics_<label>.jsonl --taxonomy output/taxonomy_<label>.jsonl --out output/report.html`

- **CLI batch run** —
  ```bash
  uv run --env-file .env -m agentic_chartqapro_eval.runner.run_generate_meps \\
      --samples src/agentic_chartqapro_eval/eval/eval_samples.json \\
      --csv "data/default of credit card clients.xls" \\
      --config anthropic_claude \\
      --n 50 --workers 4
  ```